<a href="https://colab.research.google.com/github/safdarmarwat/colab/blob/main/testing_RAG03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# The -U ensures we upgrade to the 2026 stable versions
!pip install -U -q langchain langchain-community langchain-groq langchain-huggingface \
    langchain-text-splitters pypdf faiss-cpu sentence-transformers gradio

In [11]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# These direct imports avoid the 'ModuleNotFoundError'
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Setup Models
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. Folder Access (Make sure RAG_Docs exists in your Drive!)
PDF_FOLDER = "/content/drive/MyDrive/RAG_Docs/"

def prepare_knowledge_base(folder):
    docs = []
    for file in os.listdir(folder):
        if file.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join(folder, file))
            docs.extend(loader.load())

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    chunks = splitter.split_documents(docs)
    return FAISS.from_documents(chunks, embeddings)

# Build the searchable database
vector_db = prepare_knowledge_base(PDF_FOLDER)
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# 3. The "Chain" (This replaces the need for langchain.chains)
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# This is the modern RAG pipeline:
# 1. Get Context -> 2. Format it -> 3. Pass to Prompt -> 4. Send to LLM -> 5. Parse Text
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Database and Chain are ready!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Database and Chain are ready!


In [ ]:
import gradio as gr

def ask_documents(message, history):
    try:
        # The rag_chain.invoke(message) is what talks to your PDFs
        return rag_chain.invoke(message)
    except Exception as e:
        return f"Oops! I ran into an error: {str(e)}"

# We removed the 'theme' argument to fix the TypeError
demo = gr.ChatInterface(
    fn=ask_documents,
    title="Secure PDF Chat",
    description="Ask me anything about the PDFs in your 'RAG_Docs' folder."
)

# share=True creates the public URL you can use on your phone or other tabs
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://018db5c0afe20a61b2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
